# Rotax 914 ULF - Step 1: Is anything wrong?

**In one sentence:** Looks at 128 seconds of engine data at a time and says whether a fault - in the engine or in one of its sensors - is present.

| Goes in | Comes out |
|---|---|
| A 128-second window of 29 inputs | A probability from 0 to 1 that something is wrong |

Learns from scratch - this is the first model.

Run the cells top to bottom with **Shift+Enter**. If this step was already trained with `run.py`, training just loads the saved model in a second; otherwise it trains here.

### 0. Setup

In [ ]:
# Finds the project folder wherever Jupyter was started, then loads everything.
import os, sys
d = os.path.abspath('.')
for _ in range(6):
    if os.path.exists(os.path.join(d, 'nb_helpers.py')): break
    if os.path.exists(os.path.join(d, 'validation_v4', 'nb_helpers.py')):
        d = os.path.join(d, 'validation_v4'); break
    d = os.path.dirname(d)
sys.path.insert(0, d)
from nb_helpers import *

ENGINE, PHASE = '914', 'detection'
setup(ENGINE, PHASE)

### 1. Why a healthy twin makes faults visible
Raw sensor readings move with power, altitude and weather, so a fault is easy
to miss in them. Subtracting what a **healthy** engine would read at the same
moment removes all of that. Below, one reading's residual on a healthy flight
and on a flight with a developing fault:

In [ ]:
healthy, faulty, ch = example_flights(ENGINE)
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(healthy.t / 60, healthy[ch], color=HEALTHY, lw=0.8, label="healthy engine")
ax.plot(faulty.t / 60, faulty[ch], color=FAULTY, lw=0.8, label="engine with a fault")
ax.axhline(0, color="k", lw=0.6)
ax.set_xlabel("minutes into the flight")
ax.set_ylabel(f"{pretty(ch)}\n(reading minus healthy twin)")
ax.legend(); plt.tight_layout(); plt.show()

### 2. Train the model (or load it)

In [ ]:
payload = train_or_load(ENGINE, PHASE)
print(f"Done. Trained for {payload.get('epochs_run', '?')} passes over the data.")

### 3. Scorecard
Measured on the test flights. The quality bar catches a model that did not learn - it is a safety check, not a target.

In [ ]:
scorecard(ENGINE, PHASE)

### 4. See it working on flights it never saw
**Left:** every possible alarm threshold, from strict to relaxed. Higher and
further left is better; the dotted line is guessing. **Right:** the model's
fault probability for healthy windows (blue) and faulty ones (red) - the less
they overlap, the easier the call.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
r = predict_test(ENGINE, PHASE)
y, p = r["true"].ravel(), r["pred"].ravel()
hi = r["resid"] >= np.quantile(r["resid"], 0.75)
fig, ax = plt.subplots(1, 2, figsize=(10.5, 4))
for mask, lab, col in ((np.ones_like(hi), "all windows", "#555"),
                       (hi, "engine clearly drifting from twin", FAULTY)):
    fpr, tpr, _ = roc_curve(y[mask], p[mask])
    ax[0].plot(fpr, tpr, color=col, label=f"{lab} (AUC {roc_auc_score(y[mask], p[mask]):.2f})")
ax[0].plot([0, 1], [0, 1], "k:", lw=0.8, label="guessing")
ax[0].set_xlabel("false alarms (share of healthy windows flagged)")
ax[0].set_ylabel("faults caught (share of faulty windows flagged)")
ax[0].set_title("Catch rate vs false-alarm rate"); ax[0].legend(fontsize=8)
bins = np.linspace(0, 1, 41)
ax[1].hist(p[y == 0], bins, color=HEALTHY, alpha=0.6, density=True, label="healthy windows")
ax[1].hist(p[y == 1], bins, color=FAULTY, alpha=0.6, density=True, label="faulty windows")
ax[1].set_xlabel("model's fault probability"); ax[1].set_title("How confident the model is")
ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()

### 5. What this means

In [ ]:
flag = p >= 0.5
print(f"At a 50% alarm threshold the model catches {flag[y == 1].mean():.0%} of faulty windows "
      f"and raises a false alarm on {flag[y == 0].mean():.0%} of healthy ones.")
print(f"Where the engine has clearly drifted from its twin, it catches {flag[(y == 1) & hi].mean():.0%}.")
print("The faults it misses are the ones with no visible effect yet - too early, or hidden by the "
      "engine's own controls - which a person watching the gauges would miss too.")

---
### Terms used

| Term | Meaning |
|---|---|
| **Window** | 128 seconds of flight data - the unit every model looks at. |
| **Twin** | A perfectly healthy simulated copy of the engine, flown through the exact same flight. |
| **Residual** | Real reading minus the healthy twin's reading. Near zero means the engine behaves as it should. |
| **Train / validation / test** | Flights used to learn / to pick the best version / to score it. A flight is only ever in one of them, so the score is on flights the model never saw. |
| **TBO** | Time between overhauls - the engine's scheduled life: 2000 h, or 1200 h for the 915. |
| **Warm start** | A step begins from what an earlier step already learned instead of from nothing. |